# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nadim7009/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

The paper is FlyRank's *State of AI-Driven SEO* (March 2026). It states its own limits clearly: it is observational, the health score is a FlyRank composite, ML pages are exploratory. The two questions below are the ones I would ask to understand how far each finding can be pushed, and the same ones I ask of my own work further down.

**Finding #4, "The Freshness Multiplier."** The paper reports that mature pages (365+ days old) refreshed within 30 days show a 3.2x health boost (10.7 to 34.5) and 57x more impressions (71 to 4,039).

*My question: where does the label come from, and what is the comparison group?* The health score includes impressions (30 points), position (30), CTR (20) and scroll depth (20). Impressions and position feed the score directly, so a jump in impressions and a jump in health partly measure the same thing twice. And pages that get refreshed are probably chosen because someone already saw potential in them. A before/after on the same pages, or a matched untouched group, would tell a reader how much of the 57x is the refresh and how much is selection. The paper already says that correlation does not prove causation, and the "Measure" line proposes the right comparison; this is the step I would want to see done.

**The ML appendix, 80/20 splits.** The paper describes Random Forest and Logistic Regression models evaluated on an 80/20 split of 61.8K content pieces, and labels them exploratory.

*My question: does the validation design support the claim?* If the 80/20 split is random by page, pages from the same client appear in both train and test, and a model can score well by recognising a client's habits rather than learning something that carries to a new site. A split grouped by client would show how much of the score survives. The paper keeps these pages secondary, which is the right call, and I measure the size of this exact effect on my own model in section 2.

In [1]:
# Setup: works in Colab (clones my repo) and locally (walks up to the repo root)
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/nadim7009/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import json, pathlib
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import HistGradientBoostingRegressor
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
OUT = pathlib.Path("work/outputs"); OUT.mkdir(parents=True, exist_ok=True)
FIG = pathlib.Path("work/figures"); FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# Same candidate pool as weeks 1 to 4: real position data, first two result pages, >= 500 impressions
pool = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 500)].reset_index(drop=True)
print(f"{len(pool):,} pages from {pool['client_id'].nunique()} clients")

FEATS = ["avg_position", "impressions_90d", "content_type", "main_intent", "word_count", "content_age_days"]
Y = pool["ctr"].values
W = pool["impressions_90d"].values
GROUPS = pool["client_id"]

def make_X(cols):
    X = pool[cols].copy()
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]):
            X[c] = X[c].fillna("unknown").astype("category")
    return X

def new_model():
    return HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05,
                                         categorical_features="from_dtype", random_state=42)

def wmae(y, p, w):
    return float(np.average(np.abs(y - p), weights=w))

def evaluate(cols, splitter, groups=None):
    """Return per-fold rule error, per-fold model error, and out-of-fold predictions for both."""
    X = make_X(cols)
    rule_err, model_err = [], []
    oof_rule = np.zeros(len(pool)); oof_model = np.zeros(len(pool))
    for tr, te in splitter.split(X, Y, groups):
        tier_median = pool.iloc[tr].groupby("position_tier")["ctr"].median()
        rp = pool.iloc[te]["position_tier"].map(tier_median).fillna(pool.iloc[tr]["ctr"].median()).values
        m = new_model().fit(X.iloc[tr], Y[tr], sample_weight=W[tr])
        mp = m.predict(X.iloc[te])
        oof_rule[te], oof_model[te] = rp, mp
        rule_err.append(wmae(Y[te], rp, W[te])); model_err.append(wmae(Y[te], mp, W[te]))
    return np.array(rule_err), np.array(model_err), oof_rule, oof_model

12,023 pages from 28 clients


## 2. My model under an honest split (before/after)

**Before:** a random 5-fold split, shuffled, which is what an unchecked first run would do. **After:** the same model, features and metric under a split grouped by client. Same data, same seed.

I also run the random split with `client_id` added as a feature, which is the version of the mistake that looks most innocent.

In [2]:
pool["client_id"] = pool["client_id"].astype(str)
rand = KFold(n_splits=5, shuffle=True, random_state=42)
grp = GroupKFold(n_splits=5)

r_rule, r_model, _, _ = evaluate(FEATS, rand)
g_rule, g_model, _, _ = evaluate(FEATS, grp, GROUPS)
c_rule, c_model, _, _ = evaluate(FEATS + ["client_id"], rand)

res = pd.DataFrame({
    "split": ["random 5-fold (before)", "random 5-fold + client_id as a feature", "grouped by client (after)"],
    "rule_wmae": [r_rule.mean(), c_rule.mean(), g_rule.mean()],
    "model_wmae": [r_model.mean(), c_model.mean(), g_model.mean()]})
res["model_lower_by_pct"] = 100 * (res["rule_wmae"] - res["model_wmae"]) / res["rule_wmae"]
print(res.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.4, 3.6))
x = np.arange(2); wd = 0.35
vals_rule = [r_rule.mean(), g_rule.mean()]; vals_model = [r_model.mean(), g_model.mean()]
ax.bar(x - wd/2, vals_rule, wd, label="tier-median rule", color="#aaa")
ax.bar(x + wd/2, vals_model, wd, label="boosting model", color="#1f6feb")
ax.set_xticks(x); ax.set_xticklabels(["random split", "grouped by client"])
ax.set_ylabel("weighted CTR error (pts, lower is better)"); ax.set_ylim(0, 0.33)
ax.legend(loc="upper center", ncol=2); ax.set_title("Same model, two splits")
for i in range(2):
    ax.text(x[i] - wd/2, vals_rule[i] + .004, f"{vals_rule[i]:.3f}", ha="center", fontsize=8)
    ax.text(x[i] + wd/2, vals_model[i] + .004, f"{vals_model[i]:.3f}", ha="center", fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "before_after_split.png", dpi=150); plt.close(fig)

metrics = {"before_random_split": {"rule_wmae": round(float(r_rule.mean()), 4), "model_wmae": round(float(r_model.mean()), 4)},
           "after_grouped_split": {"rule_wmae": round(float(g_rule.mean()), 4), "model_wmae": round(float(g_model.mean()), 4)},
           "random_with_client_id": {"rule_wmae": round(float(c_rule.mean()), 4), "model_wmae": round(float(c_model.mean()), 4)}}
metrics["model_gain_pct_before"] = round(100 * (r_rule.mean() - r_model.mean()) / r_rule.mean(), 1)
metrics["model_gain_pct_after"] = round(100 * (g_rule.mean() - g_model.mean()) / g_rule.mean(), 1)
json.dump(metrics, open(OUT / "validation_metrics.json", "w"), indent=2)
print(f"\nModel gain over rule: {metrics['model_gain_pct_before']}% under the random split, "
      f"{metrics['model_gain_pct_after']}% under the grouped split")

                                 split  rule_wmae  model_wmae  model_lower_by_pct
                random 5-fold (before)      0.223       0.195              12.849
random 5-fold + client_id as a feature      0.223       0.186              16.622
             grouped by client (after)      0.255       0.245               4.094

Model gain over rule: 12.8% under the random split, 4.1% under the grouped split


**Result (measured, 12,023 pages, 28 clients).**

- Random split: model error 0.195 against 0.223 for the rule, so the model looks about 13% better.
- Grouped by client: 0.245 against 0.255, about 4% better.
- Both methods score worse on the grouped split, and the model's advantage shrinks to about a third of what the random split showed. Part of what the random split rewarded was recognising a client's pages from its other pages.
- Adding `client_id` as a feature under the random split lowers the model's error further and does nothing for the rule. A model that gets better by learning the client's name has not learned something that transfers to a new client, so `client_id` stays out of the feature set.

The model's number I will report is the grouped one.

## 3. Leakage audit

Same hunt as week 3, on the final feature set. The question for each feature: would I know this value at the moment I score a page, and does it contain the answer?

In [3]:
audit = pd.DataFrame([
    ["avg_position", "Search Console, same 90d window", "known at scoring time; used for the tier too", "ok"],
    ["impressions_90d", "Search Console, same 90d window", "known at scoring time; CTR denominator, not numerator", "ok, watched"],
    ["content_type", "page metadata", "fixed trait of the page", "ok"],
    ["main_intent", "page metadata", "fixed trait of the page", "ok"],
    ["word_count", "page metadata", "fixed trait of the page", "ok"],
    ["content_age_days", "page metadata", "known at scoring time", "ok"],
    ["clicks_90d", "Search Console", "ctr = clicks / impressions * 100, so this IS the target", "REMOVED, leak"],
    ["client_id", "grouping key", "lets the model memorise a site under a random split", "REMOVED from features"],
    ["trend_direction / trend_pct", "starter label source", "built from the same pages' later movement", "never used"],
], columns=["feature", "source", "check", "verdict"])
print(audit.to_string(index=False))

# Probe 1: how close is ctr to clicks / impressions? (a near-perfect match means clicks cannot be a feature)
recon = (pool["clicks_90d"] / pool["impressions_90d"] * 100)
print(f"\nCorrelation of ctr with clicks/impressions*100: {np.corrcoef(pool['ctr'], recon)[0, 1]:.4f}")

# Probe 2: what a leaked feature does to the score (grouped split, so the only change is the leak)
l_rule, l_model, _, _ = evaluate(FEATS + ["clicks_90d"], grp, GROUPS)
print(f"Grouped split, clicks_90d added:  model error {l_model.mean():.3f}  (clean model {g_model.mean():.3f})")
print("An error this low is the signature of a leak, not a good model.")

# Probe 3: any single feature that predicts ctr suspiciously well on held-out clients?
singles = {}
for f in FEATS:
    _, e, _, _ = evaluate([f], grp, GROUPS); singles[f] = e.mean()
print("\nSingle-feature models, held-out clients (weighted error; the rule scores %.3f):" % g_rule.mean())
print(pd.Series(singles).round(3).sort_values().to_string())
assert min(singles.values()) > 0.2, "a single feature predicts too well, look for a leak"

                    feature                          source                                                   check               verdict
               avg_position Search Console, same 90d window            known at scoring time; used for the tier too                    ok
            impressions_90d Search Console, same 90d window   known at scoring time; CTR denominator, not numerator           ok, watched
               content_type                   page metadata                                 fixed trait of the page                    ok
                main_intent                   page metadata                                 fixed trait of the page                    ok
                 word_count                   page metadata                                 fixed trait of the page                    ok
           content_age_days                   page metadata                                   known at scoring time                    ok
                 clicks_90d       

**Audit result.** The current six features pass. The two things that would have leaked, `clicks_90d` and `client_id`, are out. A page's own clicks drive the error down to about 0.045, which looks wonderful and means nothing: the model is just rebuilding the target. No single clean feature gets near that, which is what I want to see.

Remaining soft spot: `impressions_90d` is the denominator of CTR, so very low-impression pages have noisy CTRs. The 500-impression floor is my guard for that, and it is a choice I made, not a proof.

### Real failure examples

The five held-out pages where the grouped model's weighted miss was largest. Public-safe columns only.

In [4]:
_, _, _, oof = evaluate(FEATS, grp, GROUPS)
pool["pred"] = oof; pool["miss_clicks"] = np.abs(pool["ctr"] - pool["pred"]) / 100 * pool["impressions_90d"]
cols = ["position_tier", "avg_position", "impressions_90d", "clicks_90d", "ctr", "pred", "content_type", "main_intent"]
worst = pool.sort_values("miss_clicks", ascending=False).head(5)[cols].round(2)
print(worst.to_string())
print(f"\nAmong the 20 largest misses: {int((pool.sort_values('miss_clicks', ascending=False).head(20)['ctr'] > pool.sort_values('miss_clicks', ascending=False).head(20)['pred']).sum())} "
      "had a CTR ABOVE the prediction (the model under-called them).")

      position_tier  avg_position  impressions_90d  clicks_90d   ctr  pred     content_type    main_intent
4291         page_1           3.3           214816        4178  1.94  0.44  keyword article  transactional
10741         top_3           2.5           509252         785  0.15  0.58  keyword article  informational
1093         page_1           7.2            62927        2138  3.40  0.19  keyword article  transactional
6760         page_1           4.8           288426        2653  0.92  0.36  keyword article  informational
5749         page_1           4.8           275226        2460  0.89  0.35  keyword article  informational

Among the 20 largest misses: 16 had a CTR ABOVE the prediction (the model under-called them).


**What the failures show.** The biggest misses are pages with very high impressions where the model predicted a typical CTR and the page did something else. These are the cases a reviewer has to see: nothing in the six features says why a page with 100,000 impressions at position 3 behaves unlike its neighbours. Branded queries, a featured snippet or an AI answer on the results page could each explain it, and this file has no query-level data to check. So a large miss is a reason to look at the page by hand, not a verdict on it.

## 4. Claim rewrite

My boldest sentence from the first run of the week-5 notebook, and the version that the evidence supports.

**Before:** "The boosting model scores pages 13% more accurately than the old rule, so ML is the better way to find pages that under-capture clicks."

**What is wrong with it:** the 13% comes from a random split, where pages from the same client sit on both sides. It says "more accurately" without naming the metric, and "better way to find pages" is a claim about the queue that I have not tested. There is no future window in this file to check that flagged pages stay low.

**After:** "On a client-held-out split of 12,023 pages from 28 clients, the model's impressions-weighted CTR error was about 0.245 against 0.255 for the tier-median rule, roughly 4% lower, and lower in 4 of 5 folds (one of those a near tie). This is a measured, directional improvement on one 90-day window. The queue it produces is decision-support for a human reviewer. It does not show that editing a flagged page will raise its clicks."

I am keeping the rule as the baseline to beat, and the next step is a future-window check on the full warehouse.

In [5]:
# The numbers quoted in the markdown above must match what this run printed.
m = json.load(open(OUT / "validation_metrics.json"))
assert abs(m["before_random_split"]["model_wmae"] - 0.195) < 0.002 and abs(m["before_random_split"]["rule_wmae"] - 0.223) < 0.002
assert abs(m["after_grouped_split"]["model_wmae"] - 0.245) < 0.002 and abs(m["after_grouped_split"]["rule_wmae"] - 0.255) < 0.002
assert int((g_model < g_rule).sum()) == 4, "the 'lower in 4 of 5 folds' sentence needs to change"
print("Claim numbers match the run.")

Claim numbers match the run.


## Self-check

- [x] Every section above is filled, with the thinking in markdown and the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime, then Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit the repo URL on the card.